In [ ]:
# =============================================================================
# Fig: 3 choropleth maps + 2 diverging bar charts  (2×2 layout)
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from shapely.geometry import box as shapely_box
from pyproj import Transformer
from adjustText import adjust_text

# ── 0. Paths & settings ───────────────────────────────────────────────────────
FLOW_DIR   = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_truncated/averaged_results/flow_avg")
LOCAL_DIR  = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/city_patient_sum")
SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
CHINA_SHP  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")
OUTFILE    = Path("/Users/shirley/Desktop/plots_V2/Fig_map_bars.png")
SCENARIO   = "earlypeak_NZ_CL"
YEARS      = [2020, 2030, 2040, 2050, 2060]

CITY_NAME_MAP = {
    "Wulumuqi":  "Urumqi",
    "Xian":      "Xi'an",
    "Qiqihaer":  "Qiqihar",
    "Huhehaote": "Hohhot",
    "Haerbin":   "Harbin",
}

C_EAST = "#C0392B"
C_WEST = "#2471A3"
PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

# ── 1. Global style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.titlesize":   10,
    "axes.titleweight": "bold",
    "axes.titlepad":    2,
})

# ── 2. Spatial data ───────────────────────────────────────────────────────────
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline   = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)

city_shp_raw = gpd.read_file(SHP_PATH)
city_shp_raw["English"] = city_shp_raw["English"].str.strip().map(
    lambda x: CITY_NAME_MAP.get(x, x))
city_shp = city_shp_raw.to_crs(PROJ_STR)

# Hu line in projected coords (compute once)
_hhy_transformer = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
_HHY_X, _HHY_Y  = _hhy_transformer.transform([127.5, 98.5], [50.2, 25.0])

# Nanhai inset bounds
_NANHAI_BOUNDS = (
    gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
    .to_crs(PROJ_STR).total_bounds
)

# Region assignment
_hhy_x0, _hhy_y0 = _HHY_X[0], _HHY_Y[0]
_hhy_x1, _hhy_y1 = _HHY_X[1], _HHY_Y[1]

def _hhy_x_at_y(y):
    t = (y - _hhy_y1) / (_hhy_y0 - _hhy_y1)
    return _hhy_x1 + t * (_hhy_x0 - _hhy_x1)

_cx = city_shp.geometry.centroid.x
_cy = city_shp.geometry.centroid.y
city_shp["region"] = np.where(_cx > _hhy_x_at_y(_cy), "East", "West")

region_map = (
    city_shp[["English", "region"]]
    .drop_duplicates(subset="English")
    .set_index("English")["region"]
    .to_dict()
)

# ── 3. Data loaders ───────────────────────────────────────────────────────────
def rename_idx(idx):
    return idx.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))

def load_flow_matrix(year):
    path = FLOW_DIR / f"flow_patientnum_avg_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path, index_col=0)
    df.index   = rename_idx(df.index)
    df.columns = rename_idx(df.columns)
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    np.fill_diagonal(df.values, 0)
    return df

def load_citysum(year):
    path = LOCAL_DIR / f"citysum_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path)
    df["city"] = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    return df.groupby("city")[["local_patient", "mo_total"]].sum()

def compute_year(year):
    df_flow  = load_flow_matrix(year)
    df_local = load_citysum(year)
    inflow   = df_flow.sum(axis=0).groupby(level=0).sum()
    outflow  = df_flow.sum(axis=1).groupby(level=0).sum()
    all_cities = inflow.index.union(outflow.index)
    inflow   = inflow.reindex(all_cities,  fill_value=0)
    outflow  = outflow.reindex(all_cities, fill_value=0)
    net      = (inflow - outflow).rename("net")
    df_local = df_local.groupby(level=0).sum()
    common   = net.index.intersection(df_local.index)
    out      = df_local.loc[common].copy()
    out["net"]    = net.loc[common].values
    out["demand"] = out["net"] + out["local_patient"]
    return out

# ── 4. Compute averages ───────────────────────────────────────────────────────
frames = [compute_year(y) for y in YEARS]
avg    = pd.concat(frames).groupby(level=0).mean()

avg["diff"]        = avg["demand"] - avg["mo_total"]
avg["net_local_r"] = avg["net"] / avg["demand"]
avg["region"]      = avg.index.map(region_map)

df_diff  = avg.sort_values("diff", ascending=False).copy()
df_ratio = (
    avg[avg["net"] > 0]
    .sort_values("net_local_r", ascending=False)
    .copy()
)

# ── 5. Merge with shapefile ───────────────────────────────────────────────────
avg_reset = (
    avg[["demand", "mo_total", "diff", "net_local_r", "region"]]
    .reset_index()
    .rename(columns={avg.index.name if avg.index.name else "index": "city"})
)
shp = city_shp.merge(avg_reset, left_on="English", right_on="city", how="left")

LABEL_THRESH = 20000

# ── 6. Nanhai inset helper ────────────────────────────────────────────────────
def _add_nanhai_inset(parent_ax, col, norm, cmap):
    x1, y1, x2, y2 = _NANHAI_BOUNDS
    axins = parent_ax.inset_axes([0.83, 0.01, 0.15, 0.18])
    axins.set_facecolor("white")
    shp.plot(column=col, ax=axins, cmap=cmap, norm=norm,
             linewidth=0, missing_kwds={"color": "lightgrey"})
    china_border.plot(ax=axins, facecolor="none", edgecolor="black", linewidth=0.2)
    jiudanline.plot(ax=axins, edgecolor="black", linewidth=0.4)
    axins.set_xlim(x1, x2)
    axins.set_ylim(y1, y2)
    axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in axins.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("black")

# ── 7. Map drawing helper ─────────────────────────────────────────────────────
def draw_map(ax, col, title, label_col, label_thresh=LABEL_THRESH):
    norm = Normalize(vmin=vmin_map, vmax=vmax_map)
    cmap = "YlOrRd"

    shp.plot(column=col, ax=ax, cmap=cmap, norm=norm,
             linewidth=0.15, edgecolor="#AAAAAA",
             missing_kwds={"color": "#DDDDDD"})
    china_border.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.15)
    jiudanline.plot(ax=ax, edgecolor="black", linewidth=0.25)

    # Hu line
    ax.plot(_HHY_X, _HHY_Y, color="black", linewidth=0.7,
            linestyle="--", dashes=(4, 3), zorder=5)
    ax.text(_HHY_X[0] + 80000, _HHY_Y[0] + 60000,
            "Hu Line", fontsize=5, color="black", zorder=5)
    
    # Clip to China bounds; trim bottom 6%
    xmin, ymin, xmax, ymax = china_border.total_bounds
    pad_x = (xmax - xmin) * 0.01
    pad_y = (ymax - ymin) * 0.01
    height = ymax - ymin
    ax.set_xlim(xmin , xmax)
    ax.set_ylim(ymin + height * 0.10, ymax + pad_y)
   
    ax.set_axis_off()

    # ── CHANGE 1: unified title via ax.text, left-aligned at axes top-left ──
    ax.text(-0.17, 1.01, title,
            transform=ax.transAxes,
            fontsize=10, fontweight="bold",
            va="bottom", ha="left")

    # City labels
    labeled = shp[shp[label_col] > label_thresh].copy()
    texts = []
    for _, row in labeled.iterrows():
        cx_r = row.geometry.centroid.x
        cy_r = row.geometry.centroid.y
        val  = row[label_col]
        txt  = ax.text(cx_r, cy_r,
                       f"{row['English']}\n({val/1000:.1f}k)",
                       fontsize=4, ha="center", va="center",
                       color="#1A1A1A", zorder=8,
                       bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                 ec="none", alpha=0.6))
        texts.append(txt)
    if texts:
        adjust_text(texts, ax=ax, time_lim=1,
                    arrowprops=dict(arrowstyle="-", color="#555555", lw=0.4))

    _add_nanhai_inset(ax, col, norm, cmap)

# ── 8. Shared norm for maps ───────────────────────────────────────────────────
vmin_map = min(shp["mo_total"].quantile(0.02), shp["demand"].quantile(0.02))
vmax_map = max(shp["mo_total"].quantile(0.98), shp["demand"].quantile(0.98))

# ── 9. Layout ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18 / 2.54, 24 / 2.54), dpi=300, facecolor="white")
gs  = GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1, 1, 0.5],
    hspace=0.22, wspace=0.20,
    left=0.06, right=0.97,
    top=0.97,   bottom=0.05,
)
ax_mapA = fig.add_subplot(gs[0, 0])
ax_mapB = fig.add_subplot(gs[0, 1])
ax_barC = fig.add_subplot(gs[1:3, 0])
ax_barD = fig.add_subplot(gs[1:3, 1])

# ── 10A & 10B. Maps ───────────────────────────────────────────────────────────
draw_map(ax_mapA, "mo_total", "a  Adaptation pressure before mobility",
         label_col="mo_total")
draw_map(ax_mapB, "demand",   "b  Adaptation pressure after mobility",
         label_col="demand")

# Shared colorbar
sm_map = mcm.ScalarMappable(
    cmap="YlOrRd", norm=Normalize(vmin=vmin_map, vmax=vmax_map))
sm_map.set_array([])
cbar_map = fig.colorbar(sm_map, ax=[ax_mapA, ax_mapB],
                        orientation="horizontal",
                        fraction=0.025, pad=0.02, shrink=0.45)
cbar_map.ax.tick_params(labelsize=6.5)
cbar_map.set_label("Patients per year (2020-2060 avg)", fontsize=6.5)

# ── 11. Bar style helper ──────────────────────────────────────────────────────
def style_bar(ax, title, xlabel):
    # ── CHANGE 1: same ax.text approach as maps, x=-0.02 so left edges align ──
    ax.text(-0.15, 1.01, title,
            transform=ax.transAxes,
            fontsize=10, fontweight="bold",
            va="bottom", ha="left")
    ax.set_xlabel(xlabel, fontsize=6.5, color="#000000")
    ax.axvline(0, color="#888", lw=0.8, ls="--", zorder=1)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0, labelsize=6.5)
    ax.tick_params(axis="x", labelsize=6)
    ax.grid(axis="x", color="#E0E0E0", lw=0.5, zorder=0)
    ax.set_facecolor("white")

# ── 12C. Bar C ────────────────────────────────────────────────────────────────
style_bar(ax_barC,
          "c  Relative pressure change (Top30)",
          "Difference in patients")

top50 = df_diff.head(30)
bot50 = df_diff.tail(30)

gap_row = pd.DataFrame(
    {col: [np.nan] for col in df_diff.columns},
    index=[""],
)
df_c = pd.concat([top50, gap_row, bot50])

n_c      = len(df_c)
y_c      = np.arange(n_c)
colors_c = [
    C_EAST if r == "East" else (C_WEST if r == "West" else "#FFFFFF")
    for r in df_c["region"]
]
ax_barC.barh(y_c, df_c["diff"].values,
             color=colors_c, height=0.78, alpha=0.88, zorder=2)
ax_barC.set_yticks(y_c)
ax_barC.set_yticklabels(df_c.index, fontsize=5)

ax_barC.axhline(30, color="#AAAAAA", lw=1.2, ls=":", zorder=3)

# ── CHANGE 2: annotations left-aligned at x=0.02 ──
ax_barC.text(0.62, 1 - 28/n_c, "▲ Top 30 inflow cities", fontsize=6.5,
             color="#555", ha="left", va="bottom",
             transform=ax_barC.transAxes)
ax_barC.text(0.62, 1 - 32/n_c, "▼ Top 30 outflow cities", fontsize=6.5,
             color="#555", ha="left", va="top",
             transform=ax_barC.transAxes)
ax_barC.invert_yaxis()

# ── 13D. Bar D ────────────────────────────────────────────────────────────────
style_bar(ax_barD,
          "d  Inflow pressure ratio",
          "Net inflow share of total demand (%)")

n_d      = len(df_ratio)
y_d      = np.arange(n_d)
colors_d = [C_EAST if r == "East" else C_WEST for r in df_ratio["region"]]
ratio_pct = df_ratio["net_local_r"].values * 100

ax_barD.barh(y_d, ratio_pct,
             color=colors_d, height=0.78, alpha=0.88, zorder=2)
ax_barD.set_yticks(y_d)
ax_barD.set_yticklabels(df_ratio.index, fontsize=5)
ax_barD.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax_barD.invert_yaxis()

# ── 14. Shared legend ─────────────────────────────────────────────────────────
patch_e = mpatches.Patch(color=C_EAST, label="East of the Hu Line", alpha=0.88)
patch_w = mpatches.Patch(color=C_WEST, label="West of the Hu Line", alpha=0.88)
fig.legend(handles=[patch_e, patch_w], loc="lower center", ncol=2,
           fontsize=6.5, frameon=False, bbox_to_anchor=(0.5, 0.0))

# ── 15. Save ──────────────────────────────────────────────────────────────────
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved → {OUTFILE}")